# Notebook 08 — Dati per l'esperimento digits (MNIST / USPS / SVHN)

Scarica ed espone in formato numpy i tre dataset usati dai notebook 09-12:

- MNIST: 60.000 train + 10.000 test, 28x28 grayscale, 10 classi
- USPS: ~7.291 train + 2.007 test, 16x16 grayscale, 10 classi
- SVHN: ~73.257 train + ~26.032 test, 32x32 RGB, 10 classi (label "10"→"0",
  già gestita internamente da `torchvision.datasets.SVHN`, verificato sotto
  invece di assunto)

Il preprocessing comune (grayscale, resize a 32x32, normalizzazione sulle
statistiche del source) avviene più avanti, in `code_v2/src/digits_data.py` -- qui
si salva solo la cache grezza convertita in `.npz`.

## Setup

In [1]:
import sys
import ssl
import shutil
import urllib.request
from pathlib import Path

here = Path().resolve()
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")

import numpy as np
from torchvision import datasets

DATA_DIR = PROJ / "data" / "digits"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJ={PROJ}")
print(f"DATA_DIR={DATA_DIR}")


def download_with_unverified_ssl(url: str, dest: Path):
    """Fallback for environments where the system CA bundle does not trust the source site."""
    with urllib.request.urlopen(url, context=ssl._create_unverified_context()) as resp, open(dest, "wb") as f:
        shutil.copyfileobj(resp, f)


def dump_split(dataset, out_path, remap_svhn_ten_to_zero: bool = False):
    X = np.stack([np.array(img) for img, _ in dataset]).astype(np.uint8)
    y = np.array([label for _, label in dataset], dtype=np.int64)
    if remap_svhn_ten_to_zero:
        # torchvision.datasets.SVHN.__init__ rimappa gi? internamente la
        # label 10 -> 0 (np.place(self.labels, self.labels == 10, 0)) --
        # verificato leggendo il suo sorgente, non assunto: qui si controlla
        # che il risultato sia davvero quello atteso, non lo si "corregge".
        assert not (y == 10).any(), "inattesa label 10 in SVHN dopo il remap interno di torchvision"
    np.savez_compressed(out_path, X=X, y=y)
    return X.shape, y.shape

def ensure_split(dataset_cls, split_kwargs, out_name: str, remap_svhn_ten_to_zero: bool = False):
    """Se out_name.npz esiste gi?, salta download e conversione (stampando
    un avviso) e ritorna le shape lette da l?; altrimenti costruisce il
    dataset (scarica se serve) e lo converte."""
    out_path = DATA_DIR / f"{out_name}.npz"
    if out_path.exists():
        print(f"{out_name}: {out_path} esiste gi? -- download e conversione saltati")
        d = np.load(out_path)
        return d["X"].shape, d["y"].shape
    ds = dataset_cls(**split_kwargs)
    return dump_split(ds, out_path, remap_svhn_ten_to_zero=remap_svhn_ten_to_zero)


PROJ=C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2
DATA_DIR=C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits


## MNIST (28x28 grayscale, 10 classi)

In [2]:
for split_name, train_flag in [("train", True), ("test", False)]:
    shape = ensure_split(datasets.MNIST, dict(root=str(RAW_DIR), train=train_flag, download=True),
                         f"mnist_{split_name}")
    print(f"mnist_{split_name}: X{shape[0]} y{shape[1]}")

mnist_train: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\mnist_train.npz esiste gi? -- download e conversione saltati


mnist_train: X(60000, 28, 28) y(60000,)
mnist_test: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\mnist_test.npz esiste gi? -- download e conversione saltati
mnist_test: X(10000, 28, 28) y(10000,)


## USPS (16x16 grayscale, 10 classi)

In [3]:
import ssl
import shutil
import urllib.request
from pathlib import Path


def download_with_unverified_ssl(url: str, dest: Path):
    """Fallback for environments where the system CA bundle does not trust the source site."""
    with urllib.request.urlopen(url, context=ssl._create_unverified_context()) as resp, open(dest, "wb") as f:
        shutil.copyfileobj(resp, f)


for split_name, train_flag in [("train", True), ("test", False)]:
    try:
        shape = ensure_split(datasets.USPS, dict(root=str(RAW_DIR), train=train_flag, download=True),
                             f"usps_{split_name}")
    except Exception as exc:
        if "CERTIFICATE_VERIFY_FAILED" not in str(exc) and "URLError" not in str(type(exc)):
            raise
        split = "train" if train_flag else "test"
        url, filename, _ = datasets.USPS.split_list[split]
        target_path = RAW_DIR / filename
        if not target_path.exists():
            print(f"usps_{split_name}: SSL certificate check failed, retrying download without verification")
            download_with_unverified_ssl(url, target_path)
        shape = ensure_split(datasets.USPS, dict(root=str(RAW_DIR), train=train_flag, download=False),
                             f"usps_{split_name}")
    print(f"usps_{split_name}: X{shape[0]} y{shape[1]}")


usps_train: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\usps_train.npz esiste gi? -- download e conversione saltati
usps_train: X(7291, 16, 16) y(7291,)
usps_test: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\usps_test.npz esiste gi? -- download e conversione saltati
usps_test: X(2007, 16, 16) y(2007,)


## SVHN (32x32 RGB, 10 classi)

`ufldl.stanford.edu` (host di SVHN) può bloccarsi a metà download (HTTP
semplice, nessun timeout nel downloader di `torchvision`). Se la cella
sotto resta ferma per più di 1-2 minuti, interrompila e riprova il file
`.mat` bloccato con un retry con ripresa, es. per `train_32x32.mat`
(ripetere per `test_32x32.mat`, dimensioni attese 182.040.794 /
64.275.384 byte):

```bash
for i in $(seq 1 15); do
  curl -fSL -C - --connect-timeout 15 --max-time 200 --speed-time 20 --speed-limit 1000 \
    -o data/digits/raw/train_32x32.mat \
    "http://ufldl.stanford.edu/housenumbers/train_32x32.mat" && break
done
```

Poi rilancia la cella sotto: `torchvision` troverà il file già scaricato e
verificato (checksum) e passerà direttamente alla conversione in `.npz`.

In [4]:
for split_name in ["train", "test"]:
    shape = ensure_split(datasets.SVHN, dict(root=str(RAW_DIR), split=split_name, download=True),
                         f"svhn_{split_name}", remap_svhn_ten_to_zero=True)
    print(f"svhn_{split_name}: X{shape[0]} y{shape[1]}")

svhn_train: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\svhn_train.npz esiste gi? -- download e conversione saltati


svhn_train: X(73257, 32, 32, 3) y(73257,)
svhn_test: C:\Users\aless\PycharmProjects\BayesianExoAdaptation\code_v2\data\digits\svhn_test.npz esiste gi? -- download e conversione saltati


svhn_test: X(26032, 32, 32, 3) y(26032,)


## Conteggi finali

In [5]:
for name in ["mnist_train", "mnist_test", "usps_train", "usps_test",
             "svhn_train", "svhn_test"]:
    d = np.load(DATA_DIR / f"{name}.npz")
    labels_present = sorted(set(d["y"].tolist()))
    print(f"  {name:12s}: {d['X'].shape[0]:6d} immagini, shape {d['X'].shape[1:]}, "
          f"label {labels_present}")

  mnist_train :  60000 immagini, shape (28, 28), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  mnist_test  :  10000 immagini, shape (28, 28), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  usps_train  :   7291 immagini, shape (16, 16), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  usps_test   :   2007 immagini, shape (16, 16), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  svhn_train  :  73257 immagini, shape (32, 32, 3), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  svhn_test   :  26032 immagini, shape (32, 32, 3), label [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
